# 04 — The estimator and its parameters: XGBoost in practice

*Chapter 09 · XGBoost · Notebook 4 of 5*

You have built XGBoost's engine by hand: the second-order leaf weight `w* = −G/(H+λ)` (NB 1), the
regularized objective and the gain that decides each split (NB 2), and the learned default direction
for missing values (NB 3). This notebook puts that engine to work through its real scikit-learn
interface, `XGBClassifier`, and turns its knobs one at a time.

Here is the honest shape of the day. XGBoost's defaults are **aggressive** — deep, fast trees that
memorize the training set out of the box. We watch them do exactly that, then learn each parameter
from the concept it controls, meet the **histogram** engine that makes the whole thing fast, and
finish by tuning the model the disciplined way: cross-validate on the training set, and open the test
set exactly once. The reward is not a dramatic accuracy jump — it is a model you understand and can
defend.

**Prerequisites.** NB 1 (`w* = −G/H`), NB 2 (the regularized objective λ/γ, the structure-score gain,
`Cover = Σ h`), NB 3 (sparsity-aware splits); Chapter 08 NB 4 (learning rate × number of trees, depth
as interaction order, the overfit); Chapter 00 (the train/test split, cross-validation, the sealed
test); Chapters 06/08 (the MDI feature-importance caveat).

**What you'll be able to do.**
- Read XGBoost's resolved defaults and say why they tend to overfit — and what defends them.
- Set each core parameter from the concept it controls (depth, the objective regularizers, stochasticity, the learning rate).
- Explain the histogram split-finder and measure its speed-up at no accuracy cost.
- Tune honestly with cross-validation and report a single sealed-test number.

## The engine, and what the knobs touch

Nothing in this notebook is new mathematics — it is the engine you already built, now driven from the
outside. Three facts from NB 1–3 carry the whole parameter set:

- every **leaf weight** is `w* = −G/(H + λ)` (NB 1–2) — the second-order step, shrunk by L2;
- every **split** is kept only if its gain `½[ G_L²/(H_L+λ) + G_R²/(H_R+λ) − G²/(H+λ) ] − γ` is
  positive (NB 2) — complexity priced inside the objective;
- every **missing row** takes a learned default direction (NB 3).

So the parameters are dials on pieces you have already seen: `reg_lambda` is the **λ** above and
`gamma` the **γ**; `min_child_weight` is a floor on a leaf's `Cover = Σ h`; `max_depth` caps how many
questions a single tree may chain — its interaction order (Chapters 04, 08). Read the knobs this way
and the documentation stops being a list of magic words.

In [ ]:
import json
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV, train_test_split
from xgboost import XGBClassifier

from ml_course import viz
from ml_course.colors import COLORS

viz.use_course_style()

SEED = 0

# A realistic tabular classification task: 20 features (8 informative, 4 redundant),
# two clusters per class, moderate separation, and flip_y = 0.04 -> 4% of the labels
# are deliberately flipped. That label noise is irreducible: no model can score above
# ~0.95-0.96 on held-out data, so memorizing the training set is a trap, not a triumph.
X, y = make_classification(
    n_samples=6000, n_features=20, n_informative=8, n_redundant=4,
    n_clusters_per_class=2, class_sep=0.9, flip_y=0.04, random_state=SEED,
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=SEED,
)
print(f"train {X_train.shape}   test {X_test.shape}")
print("class balance (train), as proportions:")
print(pd.Series(y_train).value_counts(normalize=True).round(3).to_string())


## XGBoost's defaults are aggressive

Before fitting, it is worth knowing what XGBoost will do out of the box — and how that differs from
Chapter 08's `GradientBoosting`. Two contrasts matter:

- It grows **deeper, faster** trees: `max_depth = 6` (scikit-learn's gradient boosting uses 3) and a
  learning rate `eta = 0.3` (scikit-learn uses 0.1). Deep trees taken in large steps memorize quickly.
- It turns **L2 regularization on by default**: `lambda = 1`. Scikit-learn's gradient boosting has no
  such penalty — XGBoost's philosophy is to "regularize by default." At the default value `lambda = 1`
  the effect is gentle (it shrinks each leaf weight `−G/(H + λ)` a little); we will **measure**, rather
  than assume, what it actually buys.

Let us read the resolved defaults from the engine itself, rather than trusting memory.

In [ ]:
# Fit the default estimator once; we reuse it below. Reading the engine's own
# resolved config is more honest than quoting the documentation.
clf_default = XGBClassifier(random_state=SEED)
clf_default.fit(X_train, y_train)

cfg = json.loads(clf_default.get_booster().save_config())
tp = cfg["learner"]["gradient_booster"]["tree_train_param"]
keys = ["eta", "max_depth", "min_child_weight", "subsample", "colsample_bytree",
        "lambda", "alpha", "gamma", "max_bin", "grow_policy"]
print("resolved core defaults (from booster.save_config):")
for k in keys:
    v = tp[k]
    try:
        f = float(v)
        v = int(f) if f.is_integer() else round(f, 3)
    except ValueError:
        pass
    print(f"  {k:18s} = {v}")
tm = cfg["learner"]["gradient_booster"]["gbtree_train_param"]["tree_method"]
print(f"  {'tree_method':18s} = {tm}  (-> hist)")
print(f"  {'n_estimators':18s} = {clf_default.get_booster().num_boosted_rounds()}")
print(f"  {'objective':18s} = {cfg['learner']['objective']['name']}")


In [ ]:
def train_test_leaves(model):
    """Train accuracy, test accuracy, and the total number of leaves in the ensemble."""
    tr = accuracy_score(y_train, model.predict(X_train))
    te = accuracy_score(y_test, model.predict(X_test))
    df = model.get_booster().trees_to_dataframe()
    n_leaves = int((df["Feature"] == "Leaf").sum())
    return tr, te, n_leaves


def fit_eval(**params):
    """Fit an XGBClassifier with these params (seed fixed) and return train_test_leaves."""
    return train_test_leaves(XGBClassifier(random_state=SEED, **params).fit(X_train, y_train))


tr, te, nl = train_test_leaves(clf_default)
print("defaults (depth 6, eta 0.3, 100 trees, lambda 1):")
print(f"  train acc = {tr:.4f}   test acc = {te:.4f}   gap = {tr - te:.4f}   leaves = {nl}")

clf_tamer = XGBClassifier(max_depth=3, learning_rate=0.1, n_estimators=200, random_state=SEED)
clf_tamer.fit(X_train, y_train)
tr2, te2, nl2 = train_test_leaves(clf_tamer)
print("tamer (depth 3, eta 0.1, 200 trees):")
print(f"  train acc = {tr2:.4f}   test acc = {te2:.4f}   gap = {tr2 - te2:.4f}   leaves = {nl2}")


**Read it — memorization is the tell.** The default model scores **exactly 1.0000** on the
training set: it has carved enough leaves (over 2,600) to place every one of the 4,200 training rows
correctly, the 4% of deliberately-flipped labels included. That is memorization, and the price is the
train→test gap of about 5.6 points. Yet the test accuracy (≈ 0.944) is already respectable — not
because of any one safeguard, but because the 4% label noise caps the achievable score near 0.95–0.96
and depth-6 trees already reach that ceiling. (The default `lambda = 1` does **not** rescue it: the
`reg_lambda` sweep below shows that turning L2 off leaves the test score a hair *higher*. L2's job is to
shrink leaf weights, not to lift accuracy.)

So the goal of tuning here is not to chase a huge test number. It is to **stop memorizing**: shrink the
gap, find the setting that generalizes, and end with a model whose behaviour you can explain. Notice the
tamer model already has a smaller gap (about 3 points) — but **not** a higher test score. Closing the
gap and raising the test are different things, and we keep both in view as we turn each knob.

## Knob group 1 — tree complexity: `max_depth`

The most direct complexity dial is the depth of each tree. Depth is **interaction order** (Chapters
04, 08): a depth-1 stump asks one question; a depth-`d` tree chains `d` questions, so it can express a
conjunction of `d` conditions ("feature 5 high *and* feature 12 low *and* …"). Deeper trees fit finer
structure — and, past a point, the noise. We sweep it and watch.

In [ ]:
depths = [1, 2, 3, 4, 6, 8, 10]
rows = [fit_eval(max_depth=d) for d in depths]
depth_tr = [r[0] for r in rows]
depth_te = [r[1] for r in rows]
depth_lv = [r[2] for r in rows]
for d, (a, b, c) in zip(depths, rows, strict=True):
    print(f"  depth {d:2d}:  train {a:.4f}   test {b:.4f}   gap {a - b:.4f}   leaves {c}")

i_peak = int(np.argmax(depth_te))

fig, (ax, ax2) = plt.subplots(1, 2, figsize=(12, 4.6))
ax.plot(depths, depth_tr, "o-", color=COLORS["train"], linewidth=2.2, label="train")
ax.plot(depths, depth_te, "o-", color=COLORS["test"], linewidth=2.2, label="test")
ax.plot(depths[i_peak], depth_te[i_peak], "o", color=COLORS["highlight"], markersize=15,
        zorder=5, label=f"test peak (depth {depths[i_peak]})")
ax.axvline(6, color=COLORS["muted"], linestyle="--", linewidth=1.5, label="default (depth 6)")
ax.set_xlabel("max_depth")
ax.set_ylabel("accuracy")
ax.legend(loc="center right")
ax.set_title("the depth dial: train vs test accuracy")
ax2.plot(depths, depth_lv, "o-", color=COLORS["model"], linewidth=2.2)
ax2.axvline(6, color=COLORS["muted"], linestyle="--", linewidth=1.5)
ax2.set_xlabel("max_depth")
ax2.set_ylabel("total leaves in the ensemble")
ax2.set_title("the cost in leaves")
plt.show()


**Read the figure (left).** Test accuracy climbs to a peak at **depth 4** (≈ 0.947), then drifts
slightly *down* as depth grows — while train accuracy marches to 1.0 and stays there. The default depth
6 is already **past** the peak: it pays for extra depth with thousands more leaves (right panel: about
1,200 at depth 4 → about 3,600 at depth 10) and buys no test accuracy at all. At the other end, depth 1
underfits — stumps cannot express the two-cluster-per-class interactions this data is built from. The
lesson is blunt: the default depth is tuned for large tables and speed, not for your problem. Always
sweep it.

## Knob group 2 — pricing complexity in the objective: `reg_lambda`, `gamma`, `min_child_weight`

These three are the regularized objective of NB 2, exposed as constructor arguments:

- **`reg_lambda`** is the L2 penalty **λ**. It shrinks every leaf weight `−G/(H + λ)` toward zero — a
  larger λ makes each tree timid.
- **`gamma`** (also called `min_split_loss`) is the **γ** of NB 2: the minimum gain a split must clear
  to be kept. It is pre-pruning, priced in the same units XGBoost reports as `Gain`.
- **`min_child_weight`** is a floor on a leaf's `Cover = Σ h` (NB 1–2): a leaf is allowed only if it
  carries at least that much second-order "evidence."

We sweep `gamma` and `min_child_weight` and watch the tree shrink.

In [ ]:
gammas = [0, 0.5, 1, 2, 5, 10]
g_rows = [fit_eval(gamma=gm) for gm in gammas]
g_tr = [r[0] for r in g_rows]
g_te = [r[1] for r in g_rows]
g_lv = [r[2] for r in g_rows]

mcws = [1, 5, 20, 50, 100]
m_rows = [fit_eval(min_child_weight=w) for w in mcws]
m_tr = [r[0] for r in m_rows]
m_te = [r[1] for r in m_rows]
m_lv = [r[2] for r in m_rows]

# reg_lambda (L2): off (0) and on (1) barely differ here; it only bites when large.
for lam in [0, 1, 100, 1000]:
    a, b, _ = fit_eval(reg_lambda=lam)
    print(f"  reg_lambda {lam:4d}:  train {a:.4f}   test {b:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
for ax, xs, tr_, te_, lv_, name in [
    (axes[0], gammas, g_tr, g_te, g_lv, "gamma"),
    (axes[1], mcws, m_tr, m_te, m_lv, "min_child_weight"),
]:
    axl = ax.twinx()
    axl.bar(range(len(xs)), lv_, color=COLORS["model"], alpha=0.25, width=0.6)
    axl.set_ylabel("leaves", color=COLORS["model"])
    ax.plot(range(len(xs)), tr_, "o-", color=COLORS["train"], linewidth=2.2,
            label="train", zorder=5)
    ax.plot(range(len(xs)), te_, "o-", color=COLORS["test"], linewidth=2.2,
            label="test", zorder=5)
    ax.set_xticks(range(len(xs)))
    ax.set_xticklabels([str(v) for v in xs])
    ax.set_xlabel(name)
    ax.set_ylabel("accuracy")
    ax.set_zorder(axl.get_zorder() + 1)
    ax.patch.set_visible(False)
    ax.legend(loc="lower left")
plt.show()


**Read the figure.** Both knobs do the same thing by different means: they make the trees smaller
and close the gap. Raising **`gamma`** refuses any split whose gain does not clear the bar (NB 2's γ):
the ensemble falls from about 2,600 leaves to a few hundred (264 at gamma 10), train accuracy comes off
1.0, and the gap narrows from 5.6 points to under 2. **`min_child_weight`** refuses *thin* leaves.
Recall `Cover = Σ h`, and for log-loss `h = p(1−p)`; at the start `p ≈ 0.5`, so each point contributes
about 0.25 — a floor of 1 means roughly "at least four points of evidence per leaf," a floor of 100
means "at least four hundred." (**`reg_lambda`**, the L2 penalty, is gentler still on this data — the
printed sweep shows it barely moves the score until around 100, and over-shrinking at 1000 hurts.) Push
either too far (gamma 10, min_child_weight 100) and the model underfits: the leaf bars shrink and the
test line sags with them. There is a sweet spot, and we find it by cross-validation, not by eye.

## Knob group 3 — stochasticity and the learning rate

Two more families of knob, both familiar:

- **`subsample`** trains each tree on a random fraction of the *rows* (Friedman's stochastic gradient
  boosting, 2002 — Chapter 08 NB 5), and **`colsample_bytree` / `bylevel` / `bynode`** train on a
  random fraction of the *columns*. Column subsampling is XGBoost's import from random forests
  (Chapter 06): sampling features decorrelates the trees. (Scikit-learn's gradient boosting has no
  column subsampling.)
- **`learning_rate`** (`eta`) × **`n_estimators`** is exactly Chapter 08 NB 4's trade-off: a smaller
  step needs more trees.

We re-feel the learning-rate trade-off with XGBoost, reading the test accuracy after each tree.

In [ ]:
n_max = 600
ks = [1, 5, 10, 20, 50, 100, 200, 300, 400, 500, 600]
etas = [0.3, 0.1, 0.03]
curves = {}
for eta in etas:
    m = XGBClassifier(n_estimators=n_max, learning_rate=eta, random_state=SEED)
    m.fit(X_train, y_train)
    curves[eta] = [accuracy_score(y_test, m.predict(X_test, iteration_range=(0, k))) for k in ks]
    best_k = ks[int(np.argmax(curves[eta]))]
    print(f"  eta {eta}:  best test {max(curves[eta]):.4f} @ {best_k} trees   "
          f"final @ {n_max} = {curves[eta][-1]:.4f}")

# Row and column subsampling are mild and mixed on this data:
for ss in [0.5, 1.0]:
    a = accuracy_score(y_test, XGBClassifier(subsample=ss, random_state=SEED)
                       .fit(X_train, y_train).predict(X_test))
    print(f"  subsample {ss}: test {a:.4f}")
for cs in [0.5, 1.0]:
    a = accuracy_score(y_test, XGBClassifier(colsample_bytree=cs, random_state=SEED)
                       .fit(X_train, y_train).predict(X_test))
    print(f"  colsample_bytree {cs}: test {a:.4f}")

palette = [COLORS["class_a"], COLORS["class_b"], COLORS["class_c"]]
fig, ax = plt.subplots(figsize=(8.5, 5))
for eta, col in zip(etas, palette, strict=True):
    ax.plot(ks, curves[eta], "o-", color=col, linewidth=2.2, label=f"eta = {eta}")
    i_best = int(np.argmax(curves[eta]))
    ax.plot(ks[i_best], curves[eta][i_best], "o", color=COLORS["highlight"],
            markersize=12, zorder=5)
ax.set_xscale("log")
ax.set_xlabel("number of trees (log scale)")
ax.set_ylabel("test accuracy")
ax.set_title("learning rate × number of trees (rose marks each best)")
ax.legend()
plt.show()


**Read the figure.** The three learning rates tell Chapter 08's story again. The big default step
(`eta = 0.3`, periwinkle) climbs fast and plateaus by about 50 trees, a touch below the others; the
middle step (`eta = 0.1`, amber) sits between; the smallest step (`eta = 0.03`, sage) climbs slowly but
reaches the **highest** accuracy by 600 trees — small steps explore the loss surface more carefully
before committing. The rose markers sit at each curve's best, which can fall before the last tree: past
the optimum, more trees add nothing here.

One contrast with Chapter 08: unlike the `ν = 1` case there, none of these curves **collapses** with
more trees — even 600 trees at `eta = 0.3` stays flat rather than turning down. The reason is the
metric, not a safeguard: **accuracy is coarse and bounded**. Once a point sits on the right side of the
boundary, extra trees only push its probability further from 0.5 (more confident) without flipping the
label, so the accuracy curve saturates near the noise ceiling. Chapter 08 watched a *regression* loss,
which has no such floor and could keep rising. (Row and column subsampling moved the test score by only
about half a point on this data — mild and mixed; they earn their keep on larger, noisier tables.)

## The engine that makes it fast: histogram split finding

So far, every split we have searched scanned **every distinct value** of a feature as a candidate
threshold (Chapter 04). On a few thousand rows that is fine; on the 42,000-row table below it means up
to 42,000 candidates per feature, per node, per tree — the search that dominates training time.

XGBoost's default `tree_method` (the `auto` you saw resolves to `hist` on tables this size) changes the
search. It first **bins** each feature into at most `max_bin` (256) buckets, then scans only the bucket
edges — at most 256 candidates instead of tens of thousands. The gain formula is the same one from NB 2; only the *set of thresholds it considers*
shrinks. Let us measure what that buys.

*(How the bin edges are placed is itself clever — a weighted quantile sketch that puts more edges where
the second-order weights concentrate, Chen & Guestrin §3.2–3.3. We name it here and build
histogram-based growth directly in Chapter 10.)*

In [ ]:
# A larger draw, where the threshold search actually costs something.
Xb, yb = make_classification(
    n_samples=60000, n_features=50, n_informative=20, n_redundant=10,
    n_clusters_per_class=2, class_sep=0.9, flip_y=0.03, random_state=SEED,
)
Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(
    Xb, yb, test_size=0.30, stratify=yb, random_state=SEED,
)

runs = []
for method, mb in [("exact", None), ("hist", 256), ("hist", 64)]:
    kwargs = dict(tree_method=method, n_estimators=300, random_state=SEED)
    if mb is not None:
        kwargs["max_bin"] = mb
    m = XGBClassifier(**kwargs)
    t0 = time.perf_counter()
    m.fit(Xb_tr, yb_tr)
    dt = time.perf_counter() - t0
    acc = accuracy_score(yb_te, m.predict(Xb_te))
    label = method if mb is None else f"{method} mb{mb}"
    runs.append((label, dt, acc))
    print(f"  {method:5s} max_bin={str(mb):4s}:  fit {dt:5.2f}s   test {acc:.4f}")

# Panel (a): the binning idea on one feature; (b) fit time; (c) test accuracy.
feat = Xb_tr[:, 0]
edges = np.quantile(feat, np.linspace(0, 1, 17))   # 16 illustrative bins (default is 256)

labels = [r[0] for r in runs]
times = [r[1] for r in runs]
accs = [r[2] for r in runs]
bar_cols = [COLORS["error"], COLORS["correct"], COLORS["correct"]]

fig, (axa, axb, axc) = plt.subplots(1, 3, figsize=(13.5, 4.3))
axa.hist(feat, bins=60, color=COLORS["model"], alpha=0.55)
for e in edges:
    axa.axvline(e, color=COLORS["error"], linewidth=1.0)
axa.set_xlabel("one feature's values  (red = ~16 illustrative bin edges)")
axa.set_ylabel("count")
axa.set_title("(a) every value → ≤ max_bin edges")
axb.bar(range(len(runs)), times, color=bar_cols)
axb.set_xticks(range(len(runs)))
axb.set_xticklabels(labels)
axb.set_ylabel("fit time (s)")
axb.set_title("(b) fit time (42k × 50, 300 trees)")
axc.bar(range(len(runs)), accs, color=bar_cols)
axc.set_xticks(range(len(runs)))
axc.set_xticklabels(labels)
axc.set_ylim(0.95, 0.97)
axc.set_ylabel("test accuracy")
axc.set_title("(c) test accuracy — same fits")
fig.tight_layout()
plt.show()


**Read the figure.** Binning makes the fit **several times faster** — here roughly 4 seconds for
`exact` down to under 1 second for `hist` (panel b) — with **no accuracy cost**: the two test scores
land within a hair of each other near 0.965 — `hist` is a touch higher here (panel c), within
run-to-run noise; coarse bins can also act as a mild regularizer, which is consistent with that.
Dropping to `max_bin = 64` is faster still for a sliver less
accuracy — a speed/resolution dial. This is why `hist` is the default in XGBoost 3.x and why gradient
boosting scaled to millions of rows.

Panel (a) shows the idea: a feature's values, with a handful of bin edges drawn (about 16 here for
legibility; the default is 256). The split search no longer tries every value — only these edges.

One last structural knob lives here too: **`grow_policy`**. The default `depthwise` grows a tree level
by level; `lossguide` instead expands whichever *leaf* promises the most gain next — the leaf-wise
strategy LightGBM is built around (Chapter 10). On this data the two scored about the same; we meet
leaf-wise growth properly next chapter.

## Tuning honestly: cross-validate on train, open the test once

We now have a drawer full of knobs, each helping a little. The disciplined way to combine them is the
one from Chapter 00:

1. Search over parameter combinations on the **training set only**, scoring each by
   **cross-validation**.
2. Pick the combination with the best cross-validated score.
3. Look at the **test set exactly once**, at the very end, to report an honest estimate.

Tuning *against* the test set — trying settings, peeking at the test score, repeating — is the cardinal
sin of Chapter 00: it launders the test data into training and inflates every number you report. We use
`GridSearchCV`, which runs the cross-validation for us.

In [ ]:
param_grid = {
    "max_depth": [2, 3, 4, 6],
    "learning_rate": [0.03, 0.1, 0.3],
    "n_estimators": [100, 300],
    "reg_lambda": [1, 10],
    "subsample": [0.7, 1.0],
}
search = GridSearchCV(
    XGBClassifier(random_state=SEED),
    param_grid, scoring="accuracy", cv=5, n_jobs=-1, verbose=1,
)
search.fit(X_train, y_train)

acc_default = accuracy_score(y_test, clf_default.predict(X_test))
acc_tuned = accuracy_score(y_test, search.predict(X_test))
print()
print(f"best CV params:      {search.best_params_}")
print(f"best CV accuracy:    {search.best_score_:.4f}")
print(f"tuned  sealed test:  {acc_tuned:.4f}")
print(f"default sealed test: {acc_default:.4f}")
print(f"delta (tuned - default): {acc_tuned - acc_default:+.4f}")


**Read the result.** Cross-validation searched 96 combinations and chose a configuration such as
{`max_depth` 6, `learning_rate` 0.3, `n_estimators` 300, `reg_lambda` 10, `subsample` 0.7}. On the
sealed test it scored about **0.947**, against the default's **0.944** — a gain of roughly **+0.003**.

That small number is the honest headline of the whole chapter. On this data the default was already
near the noise ceiling, so tuning bought only a sliver of test accuracy — and mostly a **more robust
model**: less memorization, a smaller gap, gentler trees. XGBoost's real advantages over a well-tuned
`GradientBoosting` are rarely a big accuracy jump; they are **speed** (the histogram engine),
**missing-value handling** (NB 3), and the **regularization** you now control. "No universal best,"
once more.

A closing caution on reading the model: `feature_importances_` here is the **gain**-based importance,
normalized to sum to 1 — the same MDI-style measure, with the same bias toward features that offer many
split points, that we flagged in Chapters 06 and 08. The trustworthy, held-out read is **permutation
importance**, which we use in the demanding case next.

## Your turn

1. **(easy)** Refit `XGBClassifier(max_depth=3, n_estimators=300)`. Before you run it, predict whether
   the train→test gap will shrink or grow versus the default — then check. (Hint: depth 3 sits below
   the test peak.)
2. **(core)** Pick **one** regularizer — `gamma` or `min_child_weight` — and sweep three values. Report
   the number of leaves and the test accuracy for each, and name which NB-2 quantity you are
   thresholding (the split gain `½[…]`, or a leaf's `Cover = Σ h`).
3. **(reach)** XGBoost's `reg_lambda` shrinks every leaf weight `−G/(H + λ)`. Using the `reg_lambda`
   sweep above, explain why turning it *up* closes the train→test gap, yet at the default `lambda = 1`
   it barely moves test accuracy on this noise-capped data — and which lever (depth, `gamma`,
   `min_child_weight`) moves the gap more. Then explain why `hist` can match `exact`'s accuracy while
   scanning far fewer thresholds — what is special about where the best split usually falls?

## What you built

You drove the real estimator. You read XGBoost's aggressive defaults from the engine, watched them
memorize the training set (train accuracy exactly 1.0), and then turned each knob from the **concept**
that owns it:

- **depth** = interaction order (`max_depth`);
- the **regularized objective** of NB 2 (`reg_lambda` = λ, `gamma` = γ, `min_child_weight` = a floor on
  `Cover = Σ h`);
- **stochasticity** (`subsample`, `colsample_*`), boosting's borrowing from random forests;
- the **learning-rate × number-of-trees** trade-off (`eta`, `n_estimators`).

You met the **histogram** split-finder that makes XGBoost fast — several times faster at no accuracy
cost — and you tuned the model the honest way: cross-validate on the training set, open the test set
once, and report a modest, robust gain.

**Vocabulary.** `tree_method='hist'` · `max_bin` · `grow_policy` (depthwise / lossguide) · `reg_lambda`
/ `gamma` / `min_child_weight` · `subsample` / `colsample_bytree` · `learning_rate` (`eta`) ×
`n_estimators` · gain (MDI) importance · the sealed test.

Next: the demanding case — a full, honest tabular workflow with early stopping, informative missing
values, and an honest cross-method comparison.

## References

- Chen, T., & Guestrin, C. (2016). *XGBoost: A Scalable Tree Boosting System.* KDD '16. DOI
  [10.1145/2939672.2939785](https://doi.org/10.1145/2939672.2939785). (§3.2–3.3, the approximate /
  histogram split-finding and the weighted quantile sketch.)
- Friedman, J. H. (2002). *Stochastic gradient boosting.* Computational Statistics & Data Analysis,
  38(4), 367–378. DOI
  [10.1016/S0167-9473(01)00065-2](https://doi.org/10.1016/S0167-9473(01)00065-2). (`subsample`.)
- Ke, G., Meng, Q., Finley, T., et al. (2017). *LightGBM: A Highly Efficient Gradient Boosting Decision
  Tree.* Advances in Neural Information Processing Systems 30 (NeurIPS 2017). (Leaf-wise growth — the
  `grow_policy='lossguide'` forward reference; Chapter 10.)
- Pedregosa, F., et al. (2011). *Scikit-learn: Machine Learning in Python.* JMLR 12, 2825–2830.
  (`GridSearchCV`.)
- Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The Elements of Statistical Learning* (2nd ed.),
  §10. DOI [10.1007/978-0-387-84858-7](https://doi.org/10.1007/978-0-387-84858-7).

*Previous: 03 — Sparsity-aware splits.*  ·  *Next: 05 — A demanding case.*